In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv("PJME_preprocessed.csv",)

In [3]:
print("Shape:", df.shape)
print("Start:", df.index.min())
print("End:", df.index.max())

Shape: (145224, 24)
Start: 0
End: 145223


In [4]:
target = "PJME_MW"

In [5]:
print(df[target].describe())

count    145224.000000
mean      32078.417972
std        6466.680687
min       14544.000000
25%       27569.000000
50%       31419.000000
75%       35647.000000
max       62009.000000
Name: PJME_MW, dtype: float64


In [6]:
print(
    "Missing values:",
    df[target].isnull().sum()
)

Missing values: 0


In [7]:
n = len(df)
test_size = int(n * 0.20)
test_start = n - test_size
print("Total observations:", n)
print("Test observations:", test_size)

Total observations: 145224
Test observations: 29044


In [8]:
validation_hours = 60 * 24
validation_start = test_start - validation_hours
print("Validation observations:", validation_hours)

Validation observations: 1440


In [9]:
train = df.iloc[:validation_start].copy()
validation = df.iloc[
    validation_start:test_start
].copy()
test = df.iloc[test_start:].copy()

In [10]:
print("Train:")
print(train.index.min(), "→", train.index.max())

print("\nValidation:")
print(validation.index.min(), "→", validation.index.max())

print("\nTest:")
print(test.index.min(), "→", test.index.max())

Train:
0 → 114739

Validation:
114740 → 116179

Test:
116180 → 145223


In [11]:
print("\nTrain size:", len(train))
print("Validation size:", len(validation))
print("Test size:", len(test))


Train size: 114740
Validation size: 1440
Test size: 29044


In [13]:
naive_validation_pred = (
    df[target]
    .shift(1)
    .loc[validation.index]
)

In [14]:
naive_test_pred = (
    df[target]
    .shift(1)
    .loc[test.index]
)

In [15]:
moving_avg = (
    df[target]
    .shift(1)
    .rolling(window=24)
    .mean()
)

In [16]:
moving_validation_pred = (moving_avg.loc[validation.index])

In [17]:
moving_test_pred = (moving_avg.loc[test.index])

In [18]:
previous_day_pred = (
    df[target]
    .shift(24)
)

In [19]:
previous_day_validation_pred = (
    previous_day_pred.loc[validation.index]
)

In [20]:
previous_day_test_pred = (
    previous_day_pred.loc[test.index]
)

In [21]:
previous_week_pred = (
    df[target]
    .shift(168)
)

In [22]:
previous_week_validation_pred = (
    previous_week_pred.loc[validation.index]
)

In [23]:
previous_week_test_pred = (
    previous_week_pred.loc[test.index]
)

In [24]:
baseline_predictions = pd.DataFrame({
    "Actual": validation[target],
    "Naive": naive_validation_pred,
    "Moving_Average": moving_validation_pred,
    "Previous_Day": previous_day_validation_pred,
    "Previous_Week": previous_week_validation_pred
})
baseline_predictions.head(20)

,Actual,Naive,Moving_Average,Previous_Day,Previous_Week
114740,39406.0,40458.0,34288.166667,33529.0,41789.0
114741,37578.0,39406.0,34533.041667,32207.0,40172.0
114742,35044.0,37578.0,34756.833333,30506.0,37764.0
114743,32804.0,35044.0,34945.916667,28728.0,35684.0
114744,31376.0,32804.0,35115.750000,27417.0,34382.0
114745,30704.0,31376.0,35280.708333,26795.0,33845.0
114746,30439.0,30704.0,35443.583333,26649.0,33722.0
114747,30627.0,30439.0,35601.500000,26935.0,33959.0
114748,31434.0,30627.0,35755.333333,28031.0,34923.0
114749,33622.0,31434.0,35897.125000,30395.0,37193.0


In [27]:
print(
    baseline_predictions.isnull().sum()
)

Actual            0
Naive             0
Moving_Average    0
Previous_Day      0
Previous_Week     0
dtype: int64


In [26]:
baseline_predictions = (
    baseline_predictions.dropna()
)

In [28]:
def calculate_metrics(actual, predicted):
    mae = mean_absolute_error(
        actual,
        predicted
    )
    mse = mean_squared_error(
        actual,
        predicted
    )
    rmse = np.sqrt(mse)
    mape = np.mean(
        np.abs(
            (actual - predicted) / actual
        )
    ) * 100
    r2 = r2_score(
        actual,
        predicted
    )
    bias = np.mean(
        predicted - actual
    )
    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [29]:
baseline_results = []
metrics = calculate_metrics(
    baseline_predictions["Actual"],
    baseline_predictions["Naive"]
)
metrics["Model"] = "Naive"
baseline_results.append(metrics)

In [30]:
metrics = calculate_metrics(
    baseline_predictions["Actual"],
    baseline_predictions["Moving_Average"]
)
metrics["Model"] = "Moving Average"
baseline_results.append(metrics)

In [31]:
metrics = calculate_metrics(
    baseline_predictions["Actual"],
    baseline_predictions["Previous_Day"]
)
metrics["Model"] = "Previous Day"
baseline_results.append(metrics)

In [32]:
metrics = calculate_metrics(
    baseline_predictions["Actual"],
    baseline_predictions["Previous_Week"]
)
metrics["Model"] = "Previous Week"
baseline_results.append(metrics)

In [33]:
baseline_results_df = pd.DataFrame(
    baseline_results
)
baseline_results_df = baseline_results_df[
    [
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Bias"
    ]
]
baseline_results_df

,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,Naive,953.077083,1.540271e+06,1241.076619,2.888684,0.949514,7.481250
1,Moving Average,2550.131973,9.774786e+06,3126.465496,7.943250,0.679610,57.267390
2,Previous Day,2459.717361,1.002802e+07,3166.705087,7.329775,0.671309,78.384028
3,Previous Week,3583.018056,1.973139e+07,4442.002433,10.922560,0.353260,939.450000


In [34]:
baseline_results_df.sort_values(
    "RMSE"
)

,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,Naive,953.077083,1.540271e+06,1241.076619,2.888684,0.949514,7.481250
1,Moving Average,2550.131973,9.774786e+06,3126.465496,7.943250,0.679610,57.267390
2,Previous Day,2459.717361,1.002802e+07,3166.705087,7.329775,0.671309,78.384028
3,Previous Week,3583.018056,1.973139e+07,4442.002433,10.922560,0.353260,939.450000


In [35]:
baseline_results_df.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,Naive,953.077083,1.540271e+06,1241.076619,2.888684,0.949514,7.481250
1,Moving Average,2550.131973,9.774786e+06,3126.465496,7.943250,0.679610,57.267390
2,Previous Day,2459.717361,1.002802e+07,3166.705087,7.329775,0.671309,78.384028
3,Previous Week,3583.018056,1.973139e+07,4442.002433,10.922560,0.353260,939.450000


In [40]:
best_baseline = baseline_results_df.loc[
    baseline_results_df["RMSE"].idxmin()
]
print(best_baseline)

Model             Naive
MAE          953.077083
MSE      1540271.175347
RMSE        1241.076619
MAPE           2.888684
R2             0.949514
Bias            7.48125
Name: 0, dtype: object


In [41]:
baseline_predictions["Previous_Day_Error"] = (
    baseline_predictions["Actual"]
    - baseline_predictions["Previous_Day"]
)

In [45]:
bias = np.mean(
    baseline_predictions["Previous_Day"]
    - baseline_predictions["Actual"]
)

print("Previous-Day Forecast Bias:", bias)

Previous-Day Forecast Bias: 78.38402777777777


In [46]:
bias_results = {
    "Naive": np.mean(
        baseline_predictions["Naive"]
        - baseline_predictions["Actual"]
    ),

    "Moving Average": np.mean(
        baseline_predictions["Moving_Average"]
        - baseline_predictions["Actual"]
    ),

    "Previous Day": np.mean(
        baseline_predictions["Previous_Day"]
        - baseline_predictions["Actual"]
    ),

    "Previous Week": np.mean(
        baseline_predictions["Previous_Week"]
        - baseline_predictions["Actual"]
    )
}
bias_df = pd.DataFrame(
    bias_results.items(),
    columns=["Model", "Bias"]
)
bias_df

,Model,Bias
0,Naive,7.481250
1,Moving Average,57.267390
2,Previous Day,78.384028
3,Previous Week,939.450000


In [47]:
baseline_results_df.to_csv("baseline_results.csv",)